In [ ]:
from fastapi import FastAPI, WebSocket, Request, BackgroundTasks, Depends, HTTPException
from fastapi.responses import HTMLResponse, StreamingResponse
from pydantic import BaseModel, Field
from typing import List, Optional
import uvicorn
import asyncio
import time
import threading
import json


In [ ]:
class Item(BaseModel):
    id: int
    name: str
    description: Optional[str] = None
    price: float


In [ ]:
app = FastAPI()

fake_db = []

@app.get("/")
def read_root():
    return {"message": "Welcome to Day 4 FastAPI"}


In [ ]:
@app.post("/items/", response_model=Item)
def create_item(item: Item):
    fake_db.append(item)
    return item

@app.get("/items/", response_model=List[Item])
def read_items():
    return fake_db

@app.get("/items/{item_id}", response_model=Item)
def read_item(item_id: int):
    for item in fake_db:
        if item.id == item_id:
            return item
    raise HTTPException(status_code=404, detail="Item not found")


In [ ]:
def write_log(message: str):
    try:
        with open("log.txt", "a") as f:
            f.write(message + "\n")
    except Exception as e:
        print(e)

@app.post("/send-notification/{email}")
def send_notification(email: str, background_tasks: BackgroundTasks):
    background_tasks.add_task(write_log, f"Notification sent to {email}")
    return {"message": "Notification sent in the background"}


In [ ]:
@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            data = await websocket.receive_text()
            await websocket.send_text(f"Message text was: {data}")
    except Exception:
        pass


In [ ]:
async def fake_video_streamer():
    for i in range(10):
        yield b"some fake video bytes\n"
        await asyncio.sleep(0.5)

@app.get("/video")
def video():
    return StreamingResponse(fake_video_streamer())


In [ ]:
if __name__ == "__main__":

    pass
